In [1]:
#匯入套件與設定環境
try:
  import ultralytics
  print("Ultralytics 已安裝，版本:", ultralytics.__version__)
except ImportError:
  print("Ultralytics 未安裝，正在安裝...")
  !pip install ultralytics
import numpy as np
import matplotlib.pyplot as plt
import torch
import os
import gdown
import shutil
import yaml
import re
from PIL import Image
from PIL import ImageDraw
from ultralytics import YOLO
from math import ceil

Ultralytics 未安裝，正在安裝...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 982.4/982.4 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 104.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-c

In [2]:
#在Colab建立資料夾
colab_save_path = "/content"

folder_name = "Where_is_Waldo"
folder_name_split = "Where_is_Waldo_split"

folder_list = ["train", "valid", "test"]
folder_list_2 = ["images", "labels"]

for folder in folder_list:
  for folder_2 in folder_list_2:
    folder_path = os.path.join(folder_name_split, folder, folder_2)
    if not os.path.exists(folder_path):
      os.makedirs(folder_path)
      print(f"建立資料夾: {folder_path}")

建立資料夾: Where_is_Waldo_split/train/images
建立資料夾: Where_is_Waldo_split/train/labels
建立資料夾: Where_is_Waldo_split/valid/images
建立資料夾: Where_is_Waldo_split/valid/labels
建立資料夾: Where_is_Waldo_split/test/images
建立資料夾: Where_is_Waldo_split/test/labels


In [3]:
#下載檔案並解壓縮
drive_url = "https://drive.google.com/file/d/1Jys9QPVqmOZbcPx8ZNK27TnMwEnqtWcd/view?usp=drive_link"
file_id = "1Jys9QPVqmOZbcPx8ZNK27TnMwEnqtWcd"
zip_url = f"https://drive.google.com/uc?id={file_id}"
zip_name = "Where_is_Waldo.zip"

gdown.download(drive_url, zip_name, fuzzy=True, quiet=False)
print(f"下載成功：{zip_name}")

shutil.unpack_archive(zip_name, folder_name)
print(f"解壓縮完成：{folder_name}")

Downloading...
From (original): https://drive.google.com/uc?id=1Jys9QPVqmOZbcPx8ZNK27TnMwEnqtWcd
From (redirected): https://drive.google.com/uc?id=1Jys9QPVqmOZbcPx8ZNK27TnMwEnqtWcd&confirm=t&uuid=3cfa21e3-9746-4cdc-a539-d2e6545dd215
To: /content/Where_is_Waldo.zip
100%|██████████| 59.0M/59.0M [00:01<00:00, 43.8MB/s]


下載成功：Where_is_Waldo.zip
解壓縮完成：Where_is_Waldo


In [4]:
#取得資料夾所有符合附檔名的檔案
def get_files_path(folder_path, file_extension):

  all_files = os.listdir(folder_path)

  files = [f for f in all_files if any(f.endswith(ext) for ext in file_extension)]

  files_path = [os.path.join(folder_path, single_file) for single_file in files]

  return files_path

In [5]:
#修改檔名
def extract_number(file_path):
  file_name = os.path.basename(file_path)
  #抓出開頭的前三位數字，如:001_jpg.rf.3e2bda47aa598c4153789cbf73a9bf83.jpg，抓出001
  match = re.match(r"(\d{3})_jpg\.rf\..+\.\w+", file_name)
  if match:
    #回傳第一個用括號括起來的部分
    return match.group(1)
  else:
    return None

for i in range(len(folder_list)):
  for j in range(len(folder_list_2)):
    folder_path = os.path.join(colab_save_path, folder_name, folder_list[i], folder_list_2[j])

    ext = ".jpg" if folder_list_2[j] == "images" else ".txt"
    all_file_paths = get_files_path(folder_path, ext)

    for file_path in all_file_paths:
      num_str = extract_number(file_path)

      #跳過不符合的檔案
      if num_str is None:
        continue

      #加上副檔名
      new_name = f"{num_str}{ext}"

      old_path = file_path
      new_path = os.path.join(folder_path, new_name)
      os.rename(old_path, new_path)

print("檔名修改成功")

檔名修改成功


In [6]:
#複製並修改yaml
source_yaml = os.path.join(colab_save_path, folder_name, "data.yaml")
dest_yaml = os.path.join(colab_save_path, folder_name_split, "data.yaml")

shutil.copy(source_yaml, dest_yaml)

#讀取
with open(dest_yaml, "r") as file:
    data = yaml.safe_load(file)

#修改yaml內的路徑
str_list = ["train", "val", "test"]

for i in range(len(str_list)):
  data[f"{str_list[i]}"] = f"{folder_list[i]}/{folder_list_2[0]}"

#寫入
with open(dest_yaml, "w") as file:
    yaml.safe_dump(data, file, default_flow_style=False)

print("yaml修改完成")
!cat "{dest_yaml}"

yaml修改完成
names:
- Waldo
nc: 1
roboflow:
  license: CC BY 4.0
  project: where-s-waldo-d1uup
  url: https://universe.roboflow.com/geass987654/where-s-waldo-d1uup/dataset/4
  version: 4
  workspace: geass987654
test: test/images
train: train/images
val: valid/images


In [7]:
#分割圖片
def generate_image_block(image, labels, block_size, overlap_ratio):

  width, height = image.size
  stride = int(block_size[0] * (1 - overlap_ratio))
  num_row = ceil((height - block_size[1]) / stride) + 1
  num_col = ceil((width - block_size[0]) / stride) + 1

  label_list = []

  #原圖中有威利(labels不為空)才處理label
  if len(labels) > 0:
    for label_str in labels:
      label = label_str.split(" ")

      #還原bounding box在原圖的座標
      label_id = int(label[0])
      label_xy = (float(label[1]) * width, float(label[2]) * height)
      label_wh = (float(label[3]) * width, float(label[4]) * height)

      label_list.append((label_id, label_xy, label_wh))

  for i in range(num_row):
    block_y1 = i * stride

    #最後一列
    if block_y1 + block_size[1] > height:
      block_y1 = height - block_size[1]

    for j in range(num_col):
      block_x1 = j * stride

      #最後一行
      if block_x1 + block_size[0] > width:
        block_x1 = width - block_size[0]

      block_x2 = block_x1 + block_size[0]
      block_y2 = block_y1 + block_size[1]

      block = image.crop((block_x1, block_y1, block_x2, block_y2))

      label_text = ""

      #原圖中有威利(label_list不為空)才調整label
      if len(label_list) > 0:
        for label_id, label_xy, label_wh in label_list:

          block_xy = (block_x1, block_y1)
          block_wh = (block_size[0], block_size[1])

          label_final = get_label_in_block(block_xy, block_wh, label_xy, label_wh, label_id, 0.25)

          if len(label_final) > 0:
            label_text += label_final + "\n"
      #儲存到Colab
      file_name = f"{img_name}_{i:02}_{j:02}"
      image_path = os.path.join(colab_save_path, folder_name_split, set_name, "images", f"{file_name}.jpg")
      label_path = os.path.join(colab_save_path, folder_name_split, set_name, "labels", f"{file_name}.txt")

      block.save(image_path)

      with open(label_path, "w") as f:
        f.write(label_text)

In [8]:
#轉換bounding box座標(原圖到區塊)
def get_label_in_block(block_xy, block_wh, label_xy, label_wh, label_id, min_visible_ratio):

  block_x , block_y = block_xy  #區塊在原圖的座標
  block_w , block_h = block_wh  #區塊寬高
  label_x , label_y = label_xy  #bounding box的中心在原圖的座標
  label_w , label_h = label_wh  #bounding box的寬高

  #左上角和右下角
  x1 = label_x - label_w / 2
  y1 = label_y - label_h / 2
  x2 = label_x + label_w / 2
  y2 = label_y + label_h / 2

  #bounding box在區塊內的座標(不超出區塊邊界)
  clipped_x1 = max(x1, block_x)
  clipped_y1 = max(y1, block_y)
  clipped_x2 = min(x2, block_x + block_w)
  clipped_y2 = min(y2, block_y + block_h)

  #bounding box面積和分割後的面積
  area = label_w * label_h
  clipped_w = max(0, clipped_x2 - clipped_x1)
  clipped_h = max(0, clipped_y2 - clipped_y1)
  clipped_area = clipped_w * clipped_h

  #原圖不存在威利，label為空

  #bounding box在區塊邊界上
  if clipped_area == 0:
    return ""

  #比例達到標準，保留bounding box
  visible_ratio = clipped_area / area

  if visible_ratio < min_visible_ratio:
    return ""

  #新bounding box的中心點(平移到原圖的左上角區塊)
  new_label_x = (clipped_x1 + clipped_x2) / 2 - block_x
  new_label_y = (clipped_y1 + clipped_y2) / 2 - block_y

  #縮放到區塊大小，介於0-1
  norm_new_label_x = round(new_label_x / block_w, 6)
  norm_new_label_y = round(new_label_y / block_h, 6)
  norm_clipped_w = round(clipped_w / block_w, 6)
  norm_clipped_h = round(clipped_h / block_h, 6)

  label_str = f"{label_id} {norm_new_label_x} {norm_new_label_y} {norm_clipped_w} {norm_clipped_h}"

  return label_str

In [16]:
# #顯示資料夾
# !ls "{colab_save_path}"

# #顯示原圖資料夾的images
!ls "{colab_save_path}/{folder_name}/{folder_list[0]}/{folder_list_2[0]}"

#顯示原圖片資料夾的labels
!ls "{colab_save_path}/{folder_name}/{folder_list[0]}/{folder_list_2[1]}"

# #顯示分割圖片資料夾的images
# !ls "{colab_save_path}/{folder_name_split}/{folder_list[0]}/{folder_list_2[0]}"

#顯示分割圖片資料夾的labels
# !ls "{colab_save_path}/{folder_name_split}/{folder_list[0]}/{folder_list_2[1]}"

# #顯示預測的txt
# !ls "runs/detect/predict/labels"

#刪除預測的結果
# !rm -r "{colab_save_path}/output"

#刪除資料夾
# !rm -r "runs/detect/train2"

# #刪除原圖資料夾
# !rm -r "{colab_save_path}/{folder_name}"

# #刪除分割資料夾
# !rm -r "{colab_save_path}/{folder_name_split}"

001.jpg  006.jpg  011.jpg  015.jpg  019.jpg  025.jpg  029.jpg  035.jpg
002.jpg  007.jpg  012.jpg  016.jpg  020.jpg  026.jpg  030.jpg  038.jpg
003.jpg  008.jpg  013.jpg  017.jpg  021.jpg  027.jpg  031.jpg
004.jpg  009.jpg  014.jpg  018.jpg  024.jpg  028.jpg  033.jpg
001.txt  006.txt  011.txt  015.txt  019.txt  025.txt  029.txt  035.txt
002.txt  007.txt  012.txt  016.txt  020.txt  026.txt  030.txt  038.txt
003.txt  008.txt  013.txt  017.txt  021.txt  027.txt  031.txt
004.txt  009.txt  014.txt  018.txt  024.txt  028.txt  033.txt


In [10]:
#圖片前處理
block_size = (640, 640)
overlap_ratio = 0.75

for i in range(len(folder_list)):
  print(f"處理資料夾: {folder_list[i]}")
  image_folder_path = os.path.join(colab_save_path, folder_name, folder_list[i], folder_list_2[0])
  label_folder_path = os.path.join(colab_save_path, folder_name, folder_list[i], folder_list_2[1])
  set_name = folder_list[i]
  image_paths = get_files_path(image_folder_path, ".jpg")

  for img_path in image_paths:
    #打開圖片
    image = Image.open(img_path)
    img_name = os.path.basename(img_path)
    img_name = os.path.splitext(img_name)[0]

    #打開label
    label_path = os.path.join(label_folder_path, f"{img_name}.txt")

    #轉換成label list(去掉頭尾的空白和換行字元)
    with open(label_path, "r") as f:
      labels = [line.strip() for line in f]

    #分割原圖
    generate_image_block(image, labels, block_size, overlap_ratio)
    print(f"{img_name} 分割完成")

處理資料夾: train
024 分割完成
031 分割完成
017 分割完成
015 分割完成
038 分割完成
009 分割完成
003 分割完成
008 分割完成
025 分割完成
020 分割完成
004 分割完成
027 分割完成
028 分割完成
035 分割完成
026 分割完成
021 分割完成
029 分割完成
011 分割完成
018 分割完成
007 分割完成
013 分割完成
033 分割完成
006 分割完成
014 分割完成
001 分割完成
002 分割完成
019 分割完成
030 分割完成
012 分割完成
016 分割完成
處理資料夾: valid
036 分割完成
023 分割完成
005 分割完成
034 分割完成
處理資料夾: test
010 分割完成
032 分割完成
022 分割完成
037 分割完成


In [11]:
#訓練模型
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_path = os.path.join(colab_save_path, folder_name_split, "data.yaml")

#代表第幾次訓練
count = 1

model = YOLO("yolov8n.pt").to(device)
result_train = model.train(data=data_path, epochs=50, imgsz=640, batch=16, device=device)

100%|██████████| 6.25M/6.25M [00:00<00:00, 107MB/s]


engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/content/Where_is_Waldo_split/data.yaml, epochs=50, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=cuda, workers=8, project=None, name=train, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, show_boxes=True, line_width=None, format=torchscript, keras=False, optimize=False, int8=False, dyn

100%|██████████| 755k/755k [00:00<00:00, 24.8MB/s]

Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics

Model summary: 129 layers, 3,011,043 parameters, 3,011,027 gradients, 8.2 GFLOPs

Transferred 319/355 items from pretrained weights
Freezing layer 'model.22.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks...


...

      50/50      3.43G     0.4725     0.3689     0.6531          0        640: 100%|██████████| 186/186 [00:48<00:00,  3.86it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 20/20 [00:03<00:00,  5.29it/s]

                   all        624         60      0.992      0.933      0.943      0.643



50 epochs completed in 0.782 hours.
Optimizer stripped from runs/detect/train/weights/last.pt, 6.2MB
Optimizer stripped from runs/detect/train/weights/best.pt, 6.2MB

Validating runs/detect/train/weights/best.pt...
Ultralytics 8.3.113 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 20/20 [00:06<00:00,  3.19it/s]


                   all        624         60      0.996      0.867      0.924      0.677
Speed: 0.8ms preprocess, 2.4ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to runs/detect/train


In [12]:
#掛載Google Drive並建立資料夾
from google.colab import drive

drive.mount('/content/drive')
drive_save_path = "/content/drive/MyDrive/WheresWaldo_with_YOLOv8"

if not os.path.exists(drive_save_path):
  os.makedirs(drive_save_path)
  print(f"建立資料夾: {drive_save_path}")

Mounted at /content/drive


In [18]:
#保存模型
from datetime import datetime
from google.colab import drive

#取得當前時間作為字串
date = datetime.now().strftime("%m%d")

model_path = os.path.join(drive_save_path, "saved_models")
best_model = os.path.join(colab_save_path, "runs", "detect", "train", "weights", "best.pt")

best_model_name = f"{date}_v{count}_best.pt"

if not os.path.exists(model_path):
  os.makedirs(model_path)
  print(f"建立資料夾: {model_path}")

shutil.copy(best_model, os.path.join(model_path, best_model_name))
print(f"保存完畢")

保存完畢


In [17]:
#用最佳模型接著訓練(訓練完要重新保存)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_path = os.path.join(colab_save_path, folder_name_split, "data.yaml")
model_path = os.path.join(drive_save_path, "saved_models")

best_model_path = os.path.join(model_path, best_model_name)

#訓練次數+1
count += 1

best_model = YOLO(best_model_path).to(device)
best_model.train(data=data_path, epochs=50, imgsz=640, batch=16, device=device)

engine/trainer: task=detect, mode=train, model=/content/drive/MyDrive/WheresWaldo_with_YOLOv8/saved_models/0422_v1_best.pt, data=/content/Where_is_Waldo_split/data.yaml, epochs=50, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=cuda, workers=8, project=None, name=train2, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, show_boxes=True, line_width=None

train: Scanning /content/Where_is_Waldo_split/train/labels.cache... 2963 images, 2638 backgrounds, 0 corrupt: 100%|██████████| 2963/2963 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 821.6±658.8 MB/s, size: 151.8 KB)


val: Scanning /content/Where_is_Waldo_split/valid/labels.cache... 624 images, 566 backgrounds, 0 corrupt: 100%|██████████| 624/624 [00:00<?, ?it/s]


...

                   all        624         60      0.954        0.8      0.899      0.679



50 epochs completed in 0.818 hours.
Optimizer stripped from runs/detect/train2/weights/last.pt, 6.2MB
Optimizer stripped from runs/detect/train2/weights/best.pt, 6.2MB

Validating runs/detect/train2/weights/best.pt...
Ultralytics 8.3.113 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 20/20 [00:05<00:00,  3.52it/s]


                   all        624         60      0.908      0.933      0.928      0.708
Speed: 0.2ms preprocess, 2.8ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to runs/detect/train2


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b934e1a5a90>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [15]:
#驗證模型
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_path = os.path.join(colab_save_path, folder_name_split, "data.yaml")
model_path = os.path.join(drive_save_path, "saved_models")

for i in range(3):
  best_model_name = f"{date}_v{count}_best.pt"
  best_model_path = os.path.join(model_path, best_model_name)
  result_val = model.val(data=data_path, epochs=50, imgsz=640, batch=16, device=device)

Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2776.7±725.0 MB/s, size: 140.9 KB)


val: Scanning /content/Where_is_Waldo_split/valid/labels.cache... 624 images, 566 backgrounds, 0 corrupt: 100%|██████████| 624/624 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 39/39 [00:09<00:00,  4.28it/s]


                   all        624         60      0.996      0.867      0.923      0.674
Speed: 2.4ms preprocess, 4.7ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to runs/detect/train2


In [ ]:
#預測模型
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_path = os.path.join(drive_save_path, "saved_models")

best_model_path = os.path.join(model_path, best_model_name)
source_path = os.path.join(colab_save_path, folder_name_split, "test", "images")

model = YOLO(best_model_path).to(device)
result_test = model.predict(source_path, conf=0.5, save=True, save_txt=True, save_conf=True, device=device)



WARNING ⚠️ inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/2963 /content/Where_is_Waldo_split/train/images/001_00_00.jpg: 640x640 (no detections), 9.6ms
image 2/2963 /content/Where_is_Waldo_split/train/images/001_00_01.jpg: 640x640 (no detections), 7.4ms
image 3/2963 /content/Where_is_Waldo_split/train/images/001_00_02.jpg: 640x640 (no detections), 7.4ms
image 4/2963 /content/Where_is_Waldo_split/train/images/001_00_03.jpg: 640x640 (no detections), 7.4ms
image 5/2963 /content/Where_is_Waldo_split/train

In [ ]:
#將預測框還原成在區塊的座標
def convert_label_to_location(txt_path, block_size):

  with open(txt_path, "r") as f:
    labels = [line.strip() for line in f]

  label_list = []

  for label_str in labels:
    label = label_str.split(" ")

    #還原座標
    id = int(label[0])
    x = int(float(label[1]) * block_size)
    y = int(float(label[2]) * block_size)
    w = int(float(label[3]) * block_size)
    h = int(float(label[4]) * block_size)
    conf= float(label[5])

    label_list.append((id, x, y, w, h, conf))

  return label_list

#計算IoU(預測框之間重疊的比例)
def compute_iou(box1, box2):

  cx1, cy1, w1, h1 = box1
  cx2, cy2, w2, h2 = box2

  #轉換為(x_min, y_min, x_max, y_max)
  x1_min, y1_min = cx1 - w1 / 2, cy1 - h1 / 2
  x1_max, y1_max = cx1 + w1 / 2, cy1 + h1 / 2

  x2_min, y2_min = cx2 - w2 / 2, cy2 - h2 / 2
  x2_max, y2_max = cx2 + w2 / 2, cy2 + h2 / 2

  #交集
  inter_x_min = max(x1_min, x2_min)
  inter_y_min = max(y1_min, y2_min)
  inter_x_max = min(x1_max, x2_max)
  inter_y_max = min(y1_max, y2_max)

  inter_width = max(0, inter_x_max - inter_x_min)
  inter_height = max(0, inter_y_max - inter_y_min)
  intersection_area = inter_width * inter_height

  #聯集
  area1 = w1 * h1
  area2 = w2 * h2
  union_area = area1 + area2 - intersection_area

  iou = intersection_area / union_area if union_area > 0 else 0.0

  return iou

#刪除重疊的預測框
def non_maximum_suppression(boxes, iou_threshold=0.5):

  #根據類別分組，避免不同類別互相影響
  grouped_boxes = {}
  for box in boxes:
    class_id = box[0]
    if class_id not in grouped_boxes:
      grouped_boxes[class_id] = []
    grouped_boxes[class_id].append(box)

  #存放最終的NMS結果
  final_boxes = []

  for class_id, class_boxes in grouped_boxes.items():
    #信心度從高排到低
    class_boxes.sort(key=lambda b: b[5], reverse=True)

    #篩選過的預測框
    selected_boxes = []

    while class_boxes:
      #取出信心度最高的框
      best_box = class_boxes.pop(0)
      selected_boxes.append(best_box)

      #過濾IoU過高的框
      class_boxes = [
        box for box in class_boxes
        if compute_iou(best_box[1:5], box[1:5]) < iou_threshold
      ]

    #加入結果
    final_boxes.extend(selected_boxes)

  return final_boxes

#刪除位於區塊邊界的預測框
def remove_box_at_boundary(boxes, block_size, border_threshold, row_index, col_index, num_row, num_col):

  boxes_new = []

  for box in boxes:

    id, x, y, w, h, conf = box

    is_near_edge = False

    x1 = x - w // 2
    y1 = y - h // 2
    x2 = x + w // 2
    y2 = y + h // 2

    #判斷是否靠近邊界
    if x1 < border_threshold or x2 > block_size - border_threshold:
      is_near_edge = True
    elif y1 < border_threshold or y2 > block_size - border_threshold:
      is_near_edge =  True

    #判斷區塊是否在第一列、第一行、最後一列、最後一行
    if is_near_edge:

      if row_index == 0 and y1 < border_threshold:
        is_near_edge = False

      elif col_index == 0 and x1 < border_threshold:
        is_near_edge = False

      elif row_index == (num_row - 1) and y2 > block_size - border_threshold:
        is_near_edge = False

      elif col_index == (num_col - 1) and x2 > block_size - border_threshold:
        is_near_edge = False

    if not is_near_edge:
      boxes_new.append(box)

  return boxes_new

In [ ]:
#顯示預測結果
import cv2
def process_images(image_folder, txt_folder, output_folder, block_size, stride):

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    #取得原圖路徑、預測的txt路徑
    image_paths = get_files_path(image_folder, ".jpg")
    txt_paths = get_files_path(txt_folder, ".txt")

    #建立txt檔案的字典
    txt_dict = {}
    for txt_path in txt_paths:
      txt_name = os.path.basename(txt_path)
      base_name = txt_name.split('_')[0]
      if base_name not in txt_dict:
        txt_dict[base_name] = []
      txt_dict[base_name].append(txt_path)

    #儲存原圖上的預測框
    all_boxes = {}

    for image_path in image_paths:
      image = cv2.imread(image_path)
      img_height, img_width = image.shape[:2]

      #原圖的檔名
      base_name = os.path.splitext(os.path.basename(image_path))[0]

      #如果沒有對應的txt檔則跳過，如圖片中沒有偵測到威利
      if base_name not in txt_dict:
        print(f"{base_name}.jpg 沒有txt檔案")
        continue

      #從檔名拆出區塊位於第幾列第幾行
      for txt_path in txt_dict[base_name]:
        txt_name = os.path.basename(txt_path)
        parts = os.path.splitext(txt_name)[0].split('_')

        if len(parts) < 3:
          print(f"{txt_name}.txt 檔名格式錯誤")

        try:
          row_index, col_index = int(parts[1].lstrip("0") or "0"), int(parts[2].lstrip("0") or "0")
        except ValueError:
          print(f"{txt_name}.txt 無法拆出row_index和col_index")
          continue

        #將預測框還原成在區塊的座標
        boxes = convert_label_to_location(txt_path, block_size)

        #刪除位於區塊邊界的預測框
        border_threshold = 10
        num_row = ceil((img_height - block_size) / stride) + 1
        num_col = ceil((img_width - block_size) / stride) + 1

        boxes = remove_box_at_boundary(boxes, block_size, border_threshold, row_index, col_index, num_row, num_col)

        #還原區塊在原圖的座標
        if row_index * stride + block_size > img_height:
          block_y = img_height - block_size
        else:
          block_y = row_index * stride

        if col_index * stride + block_size > img_width:
          block_x = img_width - block_size
        else:
          block_x = col_index * stride

        for id, x, y, w, h, conf in boxes:
          #還原預測框在原圖的座標
          orig_x, orig_y = block_x + x, block_y + y

          if base_name not in all_boxes:
              all_boxes[base_name] = []
          all_boxes[base_name].append((id, orig_x, orig_y, w, h, conf))

    #輸出圖片
    for image_path in image_paths:
      image = cv2.imread(image_path).copy()

      #原圖的檔名
      image_name = os.path.splitext(os.path.basename(image_path))[0]

      if image_name not in all_boxes:
        print(f"{image_name}.jpg 沒有偵測到威利")

      #有偵測到威利
      else:
        #合併靠近的預測框
        merged_boxes = non_maximum_suppression(all_boxes[image_name])

        #所有預測類別
        class_names = model.names

        for id, x, y, w, h, conf in merged_boxes:

          x1 = x - w // 2
          y1 = y - h // 2
          x2 = x + w // 2
          y2 = y + h // 2

          #繪製預測框
          cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 2)
          #加上文字
          text = f"{class_names[id]} {conf:.2}"
          text_size = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.9, 2)[0]
          text_width, text_height = text_size
          cv2.rectangle(image, (x1, y1 - text_height - 20), (x1 + text_width, y1), (0, 0, 255), -1)
          cv2.putText(image, text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)

      output_path = os.path.join(output_folder, image_name + "_out.jpg")
      cv2.imwrite(output_path, image)

#測試函式
process_images(
    image_folder = os.path.join(colab_save_path, folder_name, "test", "images"),
    txt_folder = os.path.join("runs", "detect", "predict", "labels"),
    output_folder = os.path.join(colab_save_path, "output"),
    block_size = 640,
    stride = 160
)
image_paths = get_files_path(os.path.join(colab_save_path, "output"), ".jpg")

#印出結果
for image_path in image_paths:
  img = Image.open(image_path)
  plt.figure(figsize=(15, 15))
  plt.imshow(img)
  plt.show()

# #儲存到Google Drive
# for image_path in image_paths:
#   shutil.copy(image_path, os.path.join(drive_save_path, "output"))

In [ ]:
#壓縮整個runs/detect資料夾並下載
from google.colab import files

!zip -r detect.zip runs/detect

files.download('detect.zip')

  adding: runs/detect/ (stored 0%)
  adding: runs/detect/predict/ (stored 0%)
  adding: runs/detect/predict/022_03_11.jpg (deflated 5%)
  adding: runs/detect/predict/037_06_02.jpg (deflated 4%)
  adding: runs/detect/predict/037_03_00.jpg (deflated 4%)
  adding: runs/detect/predict/032_01_10.jpg (deflated 6%)
  adding: runs/detect/predict/037_06_01.jpg (deflated 4%)
  adding: runs/detect/predict/010_03_02.jpg (deflated 4%)
  adding: runs/detect/predict/022_00_08.jpg (deflated 5%)
  adding: runs/detect/predict/037_00_13.jpg (deflated 4%)
  adding: runs/detect/predict/010_04_07.jpg (deflated 4%)
  adding: runs/detect/predict/037_00_04.jpg (deflated 4%)
  adding: runs/detect/predict/032_02_11.jpg (deflated 6%)
  adding: runs/detect/predict/010_01_02.jpg (deflated 4%)
  adding: runs/detect/predict/022_07_02.jpg (deflated 5%)
  adding: runs/detect/predict/010_01_03.jpg (deflated 4%)
  adding: runs/detect/predict/022_07_03.jpg (deflated 5%)
  adding: runs/detect/predict/037_08_08.jpg (deflate

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>